# 07.06 · Application structure, local server, and Studio

**Module 07 · LangGraph Fundamentals** · Part 1 (Core)

| | |
|---|---|
| ⏱️ Estimated time | 90–120 min |
| 📋 Prerequisites | Lessons 07.02–07.05; Lesson 02.05 (first `langgraph dev`). |
| 🔑 Keys | none (optional: `GROQ_API_KEY` for live models, `LANGSMITH_API_KEY` for traces) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-12 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Lay out** a LangGraph application: graph code, `langgraph.json`, dependencies, environment.
2. **Run** it with `langgraph dev` and call it over HTTP with `langgraph-sdk` (threads, runs, state history).
3. **Debug** it in Studio: step through nodes, inspect state, rerun from a checkpoint (a preview of Module 08's time travel).
4. **Read** cost and latency per node from the run itself (`run_log`) and from token usage.

## 📚 Official reading (verified 2026-10-12)

| Page | Read it for |
|---|---|
| [Application structure](https://docs.langchain.com/oss/python/langgraph/application-structure) | Files of an app, the `langgraph.json` keys (`dependencies`, `graphs`, `env`) |
| [Run a local server](https://docs.langchain.com/oss/python/langgraph/local-server) | `langgraph dev`, the Studio URL, `--tunnel` for Safari |
| [LangGraph Studio](https://docs.langchain.com/langsmith/studio) | Graph and chat modes, editing state, forking runs |
| [LangGraph CLI reference](https://docs.langchain.com/langsmith/cli) | `langgraph dev` / `validate` options |

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain-core>=1.6.9,<2" "langgraph>=1.2.14,<2" "langgraph-cli[inmem]>=0.4.33,<1" "langgraph-api>=0.16.0,<1"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."*, possibly below a few lines from pip. IPython prints that note after **every** `%pip`; restart only if a package was **upgraded** after you had already imported it (Lesson 00.05). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.02).

In [ ]:
from importlib import metadata

for package in ("langgraph", "langgraph-cli", "langgraph-api", "langgraph-sdk",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Step 1: Find the repository root (Jupyter starts in modules/07-langgraph-fundamentals/).
import importlib.util
import sys
from pathlib import Path

REPO_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone.")

# Step 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('GROQ_API_KEY', 'LANGSMITH_API_KEY')
# Step 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
from mastery.config import env_status, is_set, load_env, load_settings

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("fast      :", settings.model_fast)
print("reasoning :", settings.model_reasoning)
if not HAVE_GROQ_KEY:
    print("\n⚠️ No GROQ_API_KEY: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")

In [ ]:
# Step 4: Silence known, harmless noise so real problems stand out.
import os
import warnings

os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")          # Chroma (07.07 reuses Module 06): no usage telemetry.
# Same as Lesson 02.04: langchain-core 1.6.9/1.6.10 Pydantic serializer warning about the injected runtime.
warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)

---
## Part 1 · What an app looks like

Project 07's pipeline is already packaged as an app in `apps/m07_content_pipeline/`. Three files:

| File | Role |
|---|---|
| `pipeline.py` | Builds the graph at import time and exposes it as the variable `graph` |
| `langgraph.json` | Tells the server **where** the graph is, **what** to install, **which** `.env` to load |
| `README.md` | How to run it |

The graph's code lives in `src/mastery/graphs/content.py`, so the notebook, the tests and the server all run **the same** graph.

In [ ]:
# 🧪 Offline: look at the app's files.
import json

APP_DIR = REPO_ROOT / "apps" / "m07_content_pipeline"
for path in sorted(APP_DIR.iterdir()):
    if path.is_file():
        print(f"{path.name:<16} {path.stat().st_size:>6} bytes")
print()
print(json.dumps(json.loads((APP_DIR / "langgraph.json").read_text(encoding="utf-8")), indent=2))

**You should now see** `README.md`, `langgraph.json`, `pipeline.py` and the JSON:

- `"dependencies": ["."]`: install this folder (the course env already has `mastery`, so nothing new).
- `"graphs": {"content_pipeline": "./pipeline.py:graph"}`: graph id → `file:variable`. The id is what Studio and the SDK call.
- `"env": "../../.env"`: the repo's `.env`, relative to this file. A missing `.env` is fine (no key → offline models).

In [ ]:
# 🧪 Offline: the CLI checks the config without starting anything.
import subprocess
import sys

result = subprocess.run([sys.executable, "-m", "langgraph_cli", "validate"], cwd=APP_DIR,
                        capture_output=True, text=True, encoding="utf-8")
print((result.stdout or result.stderr).strip().splitlines()[-1])

**You should now see** `Configuration file ... is valid. (1 graph found)`. Break the graph path in a copy of the file and `validate` names the problem before you start a server.

---
## Part 2 · Run it with `langgraph dev`

**The normal way** (what you did in Lesson 02.05): a **second Git Bash window**, env active:

```bash
cd ~/langchain-langgraph-mastery/apps/m07_content_pipeline
conda activate lc-mastery
langgraph dev            # Ctrl+C to stop. Safari or Brave? add --tunnel
```

It prints the API URL (`http://127.0.0.1:2024`) and opens Studio. Keep it running while you click around.

**In this notebook** we start the same server as a child process so every cell runs with Restart & Run All: `mastery.graphs.server.DevServer` runs `python -m langgraph_cli dev --no-browser --no-reload` on a free port, waits for `/ok`, and stops it when we're done.

In [ ]:
# Parameter: False = offline scripted models in the server (free, even if you have a key).
USE_LIVE_MODELS = False

> 💸 Calls `gpt-oss-20b` 6–12 times per run (plan, write, critique, revise), a few thousand tokens of your free-tier quota. Without `GROQ_API_KEY` it prints `⏭️ Skipped`. Only when `USE_LIVE_MODELS = True`.

In [ ]:
# 🧪 Offline by default: start `langgraph dev` for the app (a local server; ~5-20 s).
from mastery.graphs.server import DevServer

LOG = REPO_ROOT / "data" / "outputs" / "m07" / "langgraph-dev.log"
live = USE_LIVE_MODELS and HAVE_GROQ_KEY
server = DevServer(APP_DIR, env={} if live else {"MASTERY_OFFLINE": "1"}, log_path=LOG).start()
print("API   :", server.url)
print("Studio:", server.studio_url)
print("Models:", "live gpt-oss-20b" if live else "offline scripted")

**You should now see** an API URL on a random port, the Studio link and `Models: offline scripted`.

Click the Studio link now (Chrome/Edge/Firefox; a free LangSmith login). You see the graph: `plan_outline` → `research_section` → `write_draft` → `critique` → `revise`/`finalize`. Type a topic in the input box and run it; watch each node light up. The server stays up until the **Stop** cell at the end of Part 4.

---
## Part 3 · Call it from code: `langgraph-sdk`

The server is an HTTP API. `langgraph-sdk` is its Python client: **assistants** (a graph + config), **threads** (a conversation's saved state), **runs** (one execution).

In [ ]:
# 🧪 Offline: one run on a thread, over HTTP.
from langgraph_sdk import get_sync_client

client = get_sync_client(url=server.url)
print("graphs :", [a["graph_id"] for a in client.assistants.search()])

thread = client.threads.create()
result = client.runs.wait(thread["thread_id"], "content_pipeline", input={"topic": "Battery care in winter"})
print("scores :", result["scores"], "| revisions:", result["iteration"])
print(result["final"][:300])

**You should now see** `graphs : ['content_pipeline']`, `scores : [4, 6, 8] | revisions: 2` and the start of the article. The server ran **your** graph; the result is the final state as JSON.

In [ ]:
# 🧪 Offline: the thread kept every checkpoint. Newest first.
history = client.threads.get_history(thread["thread_id"], limit=50)
print(len(history), "checkpoints")
for snap in history[:6]:
    print(f"next={str(snap['next']):<14} iteration={snap['values'].get('iteration')}")

**You should now see** 11 checkpoints; the newest has `next=[]` (done), then `['finalize']`, `['critique']`, `['revise']`, ... This list is what Studio shows in its thread timeline.

---
## Part 4 · Cost and latency per node

Every node in the pipeline is wrapped in `timed(...)` (from `mastery.graphs.nodes`): it appends `{node, seconds}` to `run_log`. Add `get_usage_metadata_callback` and you have tokens too.

In [ ]:
# 🧪 Offline: latency per node from the run itself.
from mastery.graphs.nodes import run_summary

print(run_summary(result["run_log"]))

**You should now see** a dict with one entry per node: calls and seconds. Offline, everything is milliseconds; `research_section` ran 3 times (one per section).

In [ ]:
# 💸 Live (in this kernel, not the server): tokens per model call.
from langchain_core.callbacks import get_usage_metadata_callback

if HAVE_GROQ_KEY:
    from mastery.graphs.content import build_content_graph, corpus_researcher
    from mastery.models import get_model

    llm = get_model("fast", temperature=0, reasoning_effort="low")
    live_graph = build_content_graph(llm, llm, llm, corpus_researcher())
    with get_usage_metadata_callback() as usage:
        live_result = live_graph.invoke({"topic": "Battery care in winter"})
    print("scores:", live_result["scores"])
    print(run_summary(live_result["run_log"]))
    for model, u in usage.usage_metadata.items():
        print(f"{model}: {u['input_tokens']} in + {u['output_tokens']} out = {u['total_tokens']} tokens")
else:
    print("⏭️ Skipped: GROQ_API_KEY is not set (see the setup cell).")

**You should now see** (with a key) live scores, seconds per node (`write_draft` and `revise` are the slow ones) and ~6–12k tokens. That's free on Groq's free tier; at paid prices it's still well under a cent.

**With `LANGSMITH_API_KEY` and `LANGSMITH_TRACING=true`** every run (in the kernel *and* in the server) is also a trace in LangSmith: the same per-node latency and tokens, clickable.

In [ ]:
# 🧪 Offline: stop the server (Ctrl+C in a terminal does the same).
server.stop()
print("stopped; log at", LOG.relative_to(REPO_ROOT))

**You should now see** `stopped; log at data/outputs/m07/langgraph-dev.log`. Open the log if a start ever fails: import errors in `pipeline.py` show up there.

---
## Part 5 · Rerun from a checkpoint (time-travel preview)

Studio's "fork" button reruns a thread from an earlier checkpoint, optionally after editing the state. The same thing in code, in-process with a checkpointer:

In [ ]:
# 🧪 Offline: run, rewind to the first critique, add a conclusion by hand, continue from there.
from mastery.graphs.content import build_content_graph, corpus_researcher, offline_models
from mastery.graphs.state import memory_checkpointer

m = offline_models()
local = build_content_graph(m["planner"], m["writer"], m["critic"], corpus_researcher(),
                            checkpointer=memory_checkpointer())
cfg = {"configurable": {"thread_id": "tt-1"}}
first = local.invoke({"topic": "Calibrating the sensors"}, cfg)
print("original scores:", first["scores"], "| revisions:", first["iteration"])

oldest_first = list(local.get_state_history(cfg))[::-1]
before_critique = next(s for s in oldest_first if s.next == ("critique",))
print("rewinding to   :", before_critique.next, "| scores so far:", before_critique.values.get("scores", []))

edited = before_critique.values["draft"] + "\n\n## Conclusion\n\nWritten by hand in the fork."
forked = local.update_state(before_critique.config, {"draft": edited})
replay = local.invoke(None, forked)
print("forked scores  :", replay["scores"], "| revisions:", replay["iteration"])

**You should now see** `original scores: [4, 6, 8] | revisions: 2`, a rewind to `('critique',)` with no scores yet, then `forked scores  : [6, 8] | revisions: 1`: the critic saw your hand-written conclusion, scored 6 instead of 4, and one revision was enough. (The revision itself rewrites the draft, so your exact sentence doesn't survive; the *effect* of the edit does.)

### What just happened?

`get_state_history` lists checkpoints; `update_state(config, values)` writes a **new** checkpoint branching from that one (the original checkpoints stay in the history); `invoke(None, config)` continues from it. Module 08 covers time travel and human-in-the-loop properly.

---
## Part 6 · Save the diagram for your README

> 🌐 Draws the graph as a PNG through mermaid.ink (a public web service; no key, no tokens). Offline or blocked? the cell saves the Mermaid text instead.

In [ ]:
# 🌐 Network: draw_mermaid_png(output_file_path=...) writes the PNG to disk.
OUT = REPO_ROOT / "data" / "outputs" / "m07"
OUT.mkdir(parents=True, exist_ok=True)
drawable = local.get_graph()
try:
    drawable.draw_mermaid_png(output_file_path=str(OUT / "content_pipeline.png"))
    print("saved", (OUT / "content_pipeline.png").relative_to(REPO_ROOT))
except Exception as exc:   # offline, rate-limited, blocked
    (OUT / "content_pipeline.mmd").write_text(drawable.draw_mermaid(), encoding="utf-8")
    print(f"⚠️ PNG failed ({type(exc).__name__}); saved", (OUT / "content_pipeline.mmd").relative_to(REPO_ROOT))

**You should now see** `saved data/outputs/m07/content_pipeline.png` (or the `.mmd` fallback). GitHub renders a ```` ```mermaid ```` block too, so the text version works in a README.

---
## ⚠️ Pitfalls and troubleshooting (Windows Git Bash included)

| Symptom | Cause | Fix |
|---|---|---|
| `langgraph: command not found` | Env not active in that window | `conda activate lc-mastery`, or `python -m langgraph_cli dev` |
| Server starts, Studio says "Failed to fetch" | Safari/Brave block `http://127.0.0.1` from an https page | `langgraph dev --tunnel`, or use Chrome/Edge/Firefox |
| `Address already in use` | An old `langgraph dev` still runs | Ctrl+C it, or `--port 2025` |
| Server keeps running after a notebook crash | The child wasn't stopped | Restart the kernel (it ends child processes) or end `python.exe` in Task Manager |
| `ModuleNotFoundError: mastery` in the log | Server started outside the course env | Activate `lc-mastery`; `python -m pip install -e ".[dev]"` |
| `UnicodeEncodeError` in the log | cp1252 console | `export PYTHONUTF8=1` (DevServer sets it for you) |

## ⚠️ Production notes

> ⚠️ **Production note: `langgraph dev` is for development.**
> It keeps state in memory and reloads on edits. Deploy with `langgraph up`/`build` (Docker) or LangSmith Deployment, which add a persistent store and task queue (Module 08).

> ⚠️ **Production note: keep secrets in `.env`, not in `langgraph.json`.**
> `langgraph.json` is committed; `"env"` only points at the file. For deployments, set secrets in the platform.

## 🏋️ Exercises

1. **(Easy)** Run `langgraph dev` in your own Git Bash window for `apps/m07_content_pipeline`, open Studio and run the topic "Mounting the station outdoors". How many revisions?
2. **(Medium)** With the SDK, create **two** threads with different topics, then list them with `client.threads.search()`. Which fields identify a thread?
3. **(Medium)** In Part 5, edit `critique` instead of `draft` (e.g. a score of 9) on the checkpoint *after* `critique`. Which node runs next?
4. **(Stretch)** Add a second graph id `"content_pipeline_strict"` to a **copy** of `langgraph.json` with `threshold=10` (a second variable in `pipeline.py`). Validate it and see both in Studio.

<details>
<summary><b>Show solutions</b></summary>

**1.** Offline models: 2 revisions, scores `[4, 6, 8]` for every topic (they're scripted).

**2.** `thread_id` (a UUID), plus `created_at`, `metadata` and `values`; `client.threads.search(limit=10)` returns dicts.

**3.** The router reads the **critique** result in state, so `finalize` runs next if you set a passing score. Editing state changes routing.

**4.** `strict_graph = build_content_graph(..., threshold=10)` and `"content_pipeline_strict": "./pipeline.py:strict_graph"`; `validate` reports `(2 graphs found)`. Offline it stops after 3 revisions.
</details>

## ✅ Checkpoint

- [ ] I can explain each key in `langgraph.json`.
- [ ] I started `langgraph dev` and ran the pipeline in Studio.
- [ ] I called the server with `langgraph-sdk` and read a thread's history.
- [ ] I can read per-node latency from `run_log`.
- [ ] Restart & Run All passes; outputs cleared before saving.

## 🔎 Research checkpoint

Read the [CLI reference](https://docs.langchain.com/langsmith/cli) for `langgraph up` and `langgraph build`, and note in [`docs/research-log.md`](../../docs/research-log.md) what they add over `langgraph dev` (Docker, a persistent store, a task queue) and what they need (Docker Desktop; check the docs for what self-hosting requires). Modules 08 and 12 build on it.

## Summary

- An app = graph code + `langgraph.json` (`dependencies`, `graphs`, `env`).
- `langgraph dev` serves it locally; Studio and `langgraph-sdk` are two clients of the same API.
- Threads keep checkpoints; `update_state` + `invoke(None, config)` reruns from one.
- `timed` nodes + `get_usage_metadata_callback` (or LangSmith) give cost and latency per node.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 07.07 · Custom agentic RAG and SQL agents in LangGraph** → [`07-07-custom-rag-and-sql-graphs.ipynb`](07-07-custom-rag-and-sql-graphs.ipynb).

[Module 07 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)